## updated harps database

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('../../DONE/archive_digger/data/toi_with_harps_v2.csv')
df.head()

In [ ]:
url = "https://github.com/3fon3fonov/HARPS_RVBank/raw/refs/heads/master/HARPS_RVBank_ver02.csv.zip"
df2 = pd.read_csv(url)
df2.head()

In [ ]:
df2.shape

In [ ]:
df2.describe()

In [ ]:
df2.dec.min()

In [ ]:
df2.shape

In [ ]:
import numpy as np

df2 = df2.replace(-9999999.0, np.nan)

# NExSci

In [ ]:
import sys
sys.path.insert(0, '..')
from wakai.catalog import get_nexsci_data

nexsci = get_nexsci_data(table_name="pscomppars", clobber=False)
nexsci.head()

In [ ]:
idx1 = nexsci.hostname.str.lower().replace('([- ])', '', regex=True).isin(df2.target.str.lower().replace('([- ])', '', regex=True))
idx2 = nexsci.hd_name.str.lower().replace('([- ])', '', regex=True).isin(df2.target.str.lower().replace('([- ])', '', regex=True))
idx3 = nexsci.hip_name.str.lower().replace('([- ])', '', regex=True).isin(df2.target.str.lower().replace('([- ])', '', regex=True))
nexsci[idx1|idx2|idx3]

In [ ]:
idx1.sum(), idx2.sum(), idx3.sum()

In [ ]:
hostnames = nexsci[idx1|idx2|idx3][['hostname','hip_name','hd_name']].stack().str.lower().replace('([- ])', '', regex=True).unique()
len(hostnames)

In [ ]:
idx=df2.target.str.lower().replace('([- ])', '', regex=True).isin(hostnames)
df2[idx].shape

In [ ]:
df2[idx].target.unique()

In [ ]:
df2.query("target=='HD133112'")

# TOI in cluster with HARPS data

In [ ]:
import pandas as pd

tois = pd.read_csv('../../toi_yuanzhe/tois_yuanzhe.txt')
tois

In [ ]:
tois = tois.TOI.apply(lambda x: int(str(x).split('.')[0])).values
tois

In [ ]:
idx = df.toi.apply(lambda x: int(x[3:])).isin(tois)
idx.sum()

## Young in nexsci

In [ ]:
nexsci_young = nexsci[nexsci.st_age<2]

In [ ]:
from astropy.coordinates import SkyCoord

d = df2.drop_duplicates(subset=['target'], keep='first').dropna(subset=['ra','dec'])
c1 = SkyCoord(d.ra.values, d.dec.values, unit='degree')

In [ ]:
c2 = SkyCoord(nexsci_young.ra.values, nexsci_young.dec.values, unit='degree')

In [ ]:
idx, d2d, _ = c1.match_to_catalog_sky(c2)

In [ ]:
import astropy.units as u

max_sep = 1.0 * u.arcsec
sep_constraint = d2d < max_sep

# Filter matched rows
cols1 = ['target']
matched_df1 = d[sep_constraint][cols1].reset_index(drop=True)
cols2 = ['hostname']
matched_df2 = nexsci_young.iloc[idx[sep_constraint]][cols2].reset_index(drop=True)

counts = df2['target'].value_counts()
matched_df1['N_RVs'] = matched_df1['target'].map(counts).fillna(0).astype(int)
matched_df = pd.concat([matched_df1, matched_df2], axis=1, keys=['df1', 'df2'])

In [ ]:
matched_df

## Any TOI

In [ ]:
import sys
sys.path.insert(0, '/ut3/jerome/github/research/project/wakai')
from wakai.catalog import get_tois

In [ ]:
tois = get_tois()

In [ ]:
from astropy.coordinates import SkyCoord

d = df2.drop_duplicates(subset=['target'], keep='first').dropna(subset=['ra','dec'])
c1 = SkyCoord(d.ra.values, d.dec.values, unit='degree')

In [ ]:
c2 = SkyCoord(tois.ra_deg.values, tois.dec_deg.values, unit='degree')

In [ ]:
idx, d2d, _ = c1.match_to_catalog_sky(c2)

In [ ]:
import astropy.units as u

max_sep = 1.0 * u.arcsec
sep_constraint = d2d < max_sep

# Filter matched rows
cols1 = ['target']
matched_df1 = d[sep_constraint][cols1].reset_index(drop=True)
cols2 = ['TOI','Comments']
matched_df2 = tois.iloc[idx[sep_constraint]][cols2].reset_index(drop=True)

counts = df2['target'].value_counts()
matched_df1['N_RVs'] = matched_df1['target'].map(counts).fillna(0).astype(int)
matched_df = pd.concat([matched_df1, matched_df2], axis=1, keys=['df1', 'df2'])

In [ ]:
matched_df.sort_values(by=('df1','N_RVs'), ascending=False)

In [ ]:
tois = tois.TOI.apply(lambda x: int(str(x).split('.')[0])).unique()
tois

In [ ]:
idx = df.toi.apply(lambda x: int(x[3:])).isin(tois)
idx.sum()